# 🕷️ Módulo 3: El Rastreador Cortés — Web Crawler
## Notebook de Conocimiento Guiado

**Objetivo:** Dominar BFS/DFS para web crawling, semáforos, deduplicación
y cortesía en la comunicación HTTP.

**Lore:** Eres el Agente Explorador de la Nebulosa Digital. Tu misión: mapear
todos los planetas web sin sobrecargar sus servidores.

| Sección | Concepto |
|---------|---------|
| 📚 Teoría | BFS vs DFS, grafos de URLs |
| 🔨 Demo | MockWeb + BFS Crawler en Python |
| ✏️ Ejercicio 1 | `BFSCrawler` con `max_depth` |
| ✏️ Ejercicio 2 | URL normalizer |
| ✏️ Ejercicio 3 | `PolitenessDelay` por dominio |
| 🎯 Quiz | Preguntas de entrevista sobre crawling |

## 📚 Parte 1: Grafos de URLs

La web es un grafo dirigido donde los nodos son páginas y las aristas son hipervínculos.

```
https://ejemplo.com        → /productos  → /laptop
                           → /blog       → /post-1
                                         → /post-2
                           → /contacto
```

**BFS** (Breadth-First Search):
- Explora por capas: primero nivel 0, luego nivel 1, luego nivel 2...
- Garantiza encontrar las URLs más cercanas primero
- **Ideal para:** indexación incremental, mapas de sitio por profundidad

**DFS** (Depth-First Search):
- Sigue un camino hasta el fondo antes de explorar otro
- Usa menos memoria si la profundidad es pequeña
- **Ideal para:** seguir cadenas de artículos relacionados

**Problema crítico:** Sin deduplicación, el crawler puede entrar en loops infinitos.
La solución es un `visited` set con acceso O(1).

In [ ]:
# 🔍 MockWeb — Simulador de internet para pruebas

class MockWeb:
    """
    Simula una 'internet' falsa con un grafo de URLs.
    Permite crawlear sin necesitar conexión real.
    """
    def __init__(self, grafo):
        # grafo: dict[str, list[str]] — URL → [URLs enlazadas]
        self.grafo = grafo
        self.peticiones = 0
    
    def fetch(self, url):
        """Simula una petición HTTP. Retorna lista de URLs encontradas."""
        self.peticiones += 1
        return self.grafo.get(url, [])
    
    def stats(self):
        return f"Total peticiones: {self.peticiones}"

# Nuestro internet simulado
internet = MockWeb({
    "https://galatik.com":          ["https://galatik.com/naves",
                                     "https://galatik.com/planetas",
                                     "https://nebula.org"],
    "https://galatik.com/naves":    ["https://galatik.com/naves/falcon",
                                     "https://galatik.com"],
    "https://galatik.com/planetas": ["https://galatik.com/planetas/tatooine"],
    "https://galatik.com/naves/falcon": [],
    "https://galatik.com/planetas/tatooine": [],
    "https://nebula.org":           ["https://nebula.org/mapa",
                                     "https://galatik.com"],
    "https://nebula.org/mapa":      [],
})
print("MockWeb creada con", len(internet.grafo), "páginas")

## 🏗️ Parte 2: BFS Crawler

In [ ]:
# 🔨 IMPLEMENTACIÓN: BFS Crawler básico
from collections import deque

def bfs_crawler(web, url_inicio, max_paginas=20):
    """
    Crawler BFS: explora la web por capas.
    
    Returns:
        list[str] — URLs visitadas en orden BFS
    """
    visitadas = set()
    cola = deque([url_inicio])
    visitadas.add(url_inicio)
    orden_visita = []
    
    while cola and len(orden_visita) < max_paginas:
        url = cola.popleft()
        
        # Simular petición HTTP
        enlaces = web.fetch(url)
        orden_visita.append(url)
        
        # Encolar nuevos enlaces no visitados
        for enlace in enlaces:
            if enlace not in visitadas:
                visitadas.add(enlace)    # Marcar ANTES de encolar (evita duplicados)
                cola.append(enlace)
    
    return orden_visita

# Comparar BFS vs DFS
def dfs_crawler(web, url_inicio, max_paginas=20):
    """Crawler DFS: usa stack en lugar de queue."""
    visitadas = set()
    stack = [url_inicio]
    visitadas.add(url_inicio)
    orden = []
    
    while stack and len(orden) < max_paginas:
        url = stack.pop()   # ← única diferencia: pop en lugar de popleft
        enlaces = web.fetch(url)
        orden.append(url)
        for enlace in reversed(enlaces):
            if enlace not in visitadas:
                visitadas.add(enlace)
                stack.append(enlace)
    
    return orden

bfs_result = bfs_crawler(internet, "https://galatik.com")
dfs_result = dfs_crawler(internet, "https://galatik.com")

print("Orden BFS:")
for i, url in enumerate(bfs_result, 1):
    print(f"  {i}. {url}")
print()
print("Orden DFS:")
for i, url in enumerate(dfs_result, 1):
    print(f"  {i}. {url}")

## 🏗️ Parte 3: Normalización de URLs y Cortesía

### URL Normalization

Dos URLs pueden referirse a la misma página:
```
https://GALATIK.COM/page     (mayúsculas)
https://galatik.com/page#section  (con fragmento)
https://galatik.com/page/       (trailing slash)
```

Debemos normalizarlas para evitar crawlear la misma página dos veces.

### Politeness Delay

Los servidores pueden bloquearte si haces demasiadas peticiones seguidas.
La solución: respetar al menos 1 segundo entre peticiones al mismo dominio.

In [ ]:
# 🔨 URL Normalization con urllib.parse
from urllib.parse import urlparse, urlunparse

def normalizar_url(url):
    """
    Normaliza una URL para deduplicación:
    - Minúsculas en scheme y host
    - Elimina fragmentos (#section)
    - Elimina query strings innecesarios
    - Normaliza trailing slash
    """
    parsed = urlparse(url)
    
    scheme = parsed.scheme.lower()
    netloc = parsed.netloc.lower()
    path = parsed.path.rstrip("/") or "/"   # remover trailing slash (excepto raíz)
    # Ignoramos: params, query, fragment (para deduplicación básica)
    
    return urlunparse((scheme, netloc, path, "", "", ""))

# Pruebas de normalización
urls_equivalentes = [
    "https://GALATIK.COM/page",
    "https://galatik.com/page/",
    "https://galatik.com/page#seccion",
    "https://galatik.com/page?utm_source=twitter",
]

print("Normalizando URLs:")
for url in urls_equivalentes:
    print(f"  {url}")
    print(f"  → {normalizar_url(url)}")
    print()

## ✏️ Ejercicio 1: BFS Crawler con `max_depth`

**Problema:** Implementa un crawler BFS que respete una profundidad máxima.
A profundidad 0 solo visita la URL de inicio. A profundidad 1, también visita
los enlaces directos. A profundidad 2, visita los enlazados desde los directos, etc.

In [ ]:
# ✏️ EJERCICIO 1 — Completa el TODO
from collections import deque

def bfs_crawler_depth(web, url_inicio, max_depth=2):
    """
    Crawler BFS que respeta profundidad máxima.
    
    Returns:
        dict[str, int] — {url: profundidad_en_la_que_se_visitó}
    """
    # TODO: Implementar BFS con control de profundidad
    # Pista: guarda (url, profundidad) en la cola
    # No sigas enlaces de páginas a profundidad == max_depth
    pass

resultado = bfs_crawler_depth(internet, "https://galatik.com", max_depth=1)
print("Visitadas a profundidad <= 1:")
for url, prof in sorted(resultado.items(), key=lambda x: x[1]):
    print(f"  profundidad {prof}: {url}")

In [ ]:
# 💡 SOLUCIÓN — Ejercicio 1

def bfs_crawler_depth(web, url_inicio, max_depth=2):
    visitadas = {}
    cola = deque([(url_inicio, 0)])
    visitadas[url_inicio] = 0
    
    while cola:
        url, profundidad = cola.popleft()
        
        if profundidad < max_depth:
            for enlace in web.fetch(url):
                norm = normalizar_url(enlace)
                if norm not in visitadas:
                    visitadas[norm] = profundidad + 1
                    cola.append((norm, profundidad + 1))
        else:
            web.fetch(url)   # Contabilizar la petición, no seguir enlaces
    
    return visitadas

resultado = bfs_crawler_depth(internet, "https://galatik.com", max_depth=1)
print("Visitadas a profundidad <= 1:")
for url, prof in sorted(resultado.items(), key=lambda x: x[1]):
    print(f"  profundidad {prof}: {url}")

## ✏️ Ejercicio 2: PolitenessDelay

**Problema:** Implementa un controlador de cortesía que garantice al menos
`min_delay` segundos entre peticiones al mismo dominio.

In [ ]:
# ✏️ EJERCICIO 2 — Completa el TODO
import time
from urllib.parse import urlparse

class PolitenessDelay:
    """Asegura que no se hagan peticiones a un dominio demasiado rápido."""
    
    def __init__(self, min_delay=1.0):
        self.min_delay = min_delay
        # TODO: Crear estructura para recordar la última petición por dominio
    
    def wait_if_needed(self, url):
        """Espera si es necesario antes de hacer petición a esta URL."""
        dominio = urlparse(url).netloc
        # TODO: Implementar la lógica de espera
        pass

# Test
delay = PolitenessDelay(min_delay=0.1)
urls = ["https://galatik.com", "https://galatik.com/naves", "https://nebula.org"]
for url in urls:
    inicio = time.time()
    delay.wait_if_needed(url)
    print(f"Petición a {url} — tiempo de espera: {time.time()-inicio:.3f}s")

In [ ]:
# 💡 SOLUCIÓN — Ejercicio 2

class PolitenessDelay:
    def __init__(self, min_delay=1.0):
        self.min_delay = min_delay
        self._ultima_peticion = {}   # dominio → timestamp
    
    def wait_if_needed(self, url):
        dominio = urlparse(url).netloc
        if dominio in self._ultima_peticion:
            transcurrido = time.time() - self._ultima_peticion[dominio]
            if transcurrido < self.min_delay:
                time.sleep(self.min_delay - transcurrido)
        self._ultima_peticion[dominio] = time.time()

delay = PolitenessDelay(min_delay=0.1)
urls = ["https://galatik.com", "https://galatik.com/naves", "https://nebula.org"]
print("Con delay de 0.1s por dominio:")
for url in urls:
    inicio = time.time()
    delay.wait_if_needed(url)
    print(f"  {url} — espera: {time.time()-inicio:.3f}s")

## 🎯 Quiz — Preguntas de Entrevista

**P1:** ¿Por qué usamos un `set` para las URLs visitadas y no una lista?
> **R:** El `set` tiene búsqueda O(1) promedio vs O(n) para la lista.
> Con millones de URLs, la diferencia es crítica.

**P2:** ¿Cuál es el problema de marcar una URL como visitada DESPUÉS de desencolarla?
> **R:** Si la misma URL aparece en varios enlaces, podría añadirse a la cola varias veces
> antes de procesarse, causando trabajo duplicado. Siempre marcar al encolar.

**P3:** ¿Cómo escalarías un crawler de 1 máquina a 1000 máquinas?
> **R:** Cola distribuida (Redis, Kafka), frontier de URLs particionada por dominio (consistent hashing),
> bloom filter para deduplicación aproximada, coordination con ZooKeeper/etcd.

**P4:** ¿Qué es el trap de las URL dinámicas?
> **R:** URLs infinitas generadas con parámetros (calendarios, filtros de búsqueda).
> Solución: normalizar, ignorar ciertos query params, detectar patrones repetitivos.